In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.config import PROCESSED, SPLITS, N_FOLDS, N_REPEATS
from src.data import load_raw, feature_cols, split_holdout_by_group
from src.groups import similarity_matrix, make_groups
from src.cv import make_group_fold_table

df = load_raw()                                    # all 500 segments, classes 1 to 5
cols = feature_cols(df)
names, M = similarity_matrix(df, cols)             # signals only: no labels, no scoring
group_of = make_groups(names, M)
cls = df.groupby("segment")["y"].first().loc[names]

sizes = group_of.value_counts()
print("segments:", len(names), "| groups:", group_of.nunique(), "| group sizes:", sizes.value_counts().sort_index().to_dict())
print("groups with more than one class:", sum(cls[group_of[group_of == g].index].nunique() > 1 for g in sizes.index))
print("groups mixing seizure and non-seizure:", sum((cls[group_of[group_of == g].index] == 1).nunique() > 1 for g in sizes.index))

segments: 500 | groups: 330 | group sizes: {1: 224, 2: 71, 3: 25, 4: 3, 5: 3, 7: 2, 9: 2}
groups with more than one class: 1


groups mixing seizure and non-seizure: 0


In [2]:
dev, hold = split_holdout_by_group(df, group_of)
per_class = lambda d: d.groupby("segment")["y"].first().value_counts().sort_index().to_dict()
print("dev segments:", dev["segment"].nunique(), per_class(dev))
print("hold-out segments:", hold["segment"].nunique(), per_class(hold), "| hold-out groups:", group_of[hold["segment"].unique()].nunique())

idx = {n: i for i, n in enumerate(names)}
h = [idx[s] for s in hold["segment"].unique()]
d = [idx[s] for s in dev["segment"].unique()]
print("most similar hold-out/dev pair:", round(M[np.ix_(h, d)].max(), 2))

dev segments: 400 {1: 80, 2: 80, 3: 80, 4: 80, 5: 80}
hold-out segments: 100 {1: 20, 2: 20, 3: 20, 4: 20, 5: 20} | hold-out groups: 66
most similar hold-out/dev pair: 0.67


In [3]:
table = make_group_fold_table(dev, group_of)
for r in range(N_REPEATS):
    col = f"repeat_{r}"
    worst = max(M[np.ix_([idx[s] for s in table["segment"][table[col] == k]],
                         [idx[s] for s in table["segment"][table[col] != k]])].max()
                for k in range(N_FOLDS))
    per_fold = table.groupby(col).size()
    seizure = table[table["y"] == 1].groupby(col).size()
    print(f"repeat {r}: most similar validation/training pair {worst:.2f} | segments per fold {per_fold.min()}-{per_fold.max()} | seizure segments per fold {seizure.min()}-{seizure.max()}")

repeat 0: most similar validation/training pair 0.64 | segments per fold 39-41 | seizure segments per fold 8-8
repeat 1: most similar validation/training pair 0.64 | segments per fold 39-41 | seizure segments per fold 8-8
repeat 2: most similar validation/training pair 0.64 | segments per fold 39-41 | seizure segments per fold 8-8
repeat 3: most similar validation/training pair 0.63 | segments per fold 39-41 | seizure segments per fold 8-8
repeat 4: most similar validation/training pair 0.64 | segments per fold 39-41 | seizure segments per fold 8-8


In [4]:
if (SPLITS / "cv_folds_allclass.csv").exists():
    print("seizure splits already exist, not overwriting")
else:
    dev.to_csv(PROCESSED / "dev_allclass.csv", index=False)
    hold.to_csv(PROCESSED / "holdout_allclass.csv", index=False)
    table.to_csv(SPLITS / "cv_folds_allclass.csv", index=False)
    pd.DataFrame({"segment": sorted(hold["segment"].unique())}).to_csv(SPLITS / "holdout_segments_allclass.csv", index=False)
    group_of.rename_axis("segment").reset_index().to_csv(SPLITS / "groups_all.csv", index=False)
    print("saved")

seizure splits already exist, not overwriting


In [5]:
t = pd.DataFrame({"class": cls.values, "group": group_of.values, "group_size": group_of.map(group_of.value_counts()).values})
t.groupby("class").agg(segments=("group", "size"), groups=("group", "nunique"),
                       biggest_group=("group_size", "max"))

,segments,groups,biggest_group
class,,,
1,100,67,7
2,100,46,9
3,100,49,9
4,100,82,2
5,100,87,2


# 08 Splits for seizure detection (class 1 vs the rest)

Twin detection and grouping on all 500 segments (signals only, no labels). Hold-out and folds are made from whole groups, stratified by the 5-class label, so every class contributes equally.

## Groups
- 500 segments form 330 groups (sizes 1 to 9). One group contains more than one class. No group mixes seizure and non-seizure segments.
- Independent groups per class: seizure 67, class 2: 46, class 3: 49, class 4: 82, class 5: 87. Classes 2 and 3 have the most twins, so fewer independent recordings than segments.

## Splits
- Hold-out: 100 segments (20 per class) from 66 groups, locked until the final test.
- Dev: 400 segments (80 per class). 10 folds x 5 repeats, 39 to 41 segments per fold, 8 seizure segments in every fold.
- Audit: the most similar hold-out/dev pair is 0.67, and the most similar validation/training pair in any fold is 0.63 to 0.64. Both are below 0.7.
- Saved in `splits/` with a `_seizure` suffix. The 2 vs 3 files are untouched.

## Caveats
- Only 67 independent seizure groups exist, and the hold-out holds 20 seizure segments, so recall will have a wide range.
- Weaker similarity below the thresholds may remain, so scores are upper bounds.